## **🟣 1: Install and Import Dependencies**
In this section, we install the necessary packages for chemical data processing and machine learning.

In [ ]:
# Install the requirements:
!pip install pandas numpy lightgbm rdkit matplotlib

In [ ]:
# Import libraries:
import pandas as pd
import numpy as np
import gcsfs
import pandas as pd
import numpy as np
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem
import os
from helper import Dataset,ProcessData, SimplifiedDrugFilters,ResultSubmission


# 🚀**Aircheck Workshop: Using Machine Learning to Find Hits**🧬
Welcome to the Aircheck ML Model Training and Prediction Notebook! This notebook walks you through the various steps involved in training, evaluating, and screening small molecules using machine learning models built on chemical fingerprints.


In [ ]:
from helper import display_google_drive_image
display_google_drive_image('1V_z04QE3QrXYPp0OpbRJmZpsVtKaQvOC')

--------------------------------------------------------------------------------

---

## **🟣 2: Load and Prepare Data**

Next, we will load datasets

**Available datasets**
Train:
- TrainDataset_Aircheck.parquet
- TrainDataset_Aircheck_class0_1x.parquet
- TrainDataset_Aircheck_class0_2x.parquet
- TrainDataset_Aircheck_class0_4x.parquet
- TrainDataset_Aircheck_class0_6x.parquet

Development:
- DevDataset_Aircheck.csv

Test:
- TestDataset_Aircheck.csv

In [ ]:
df_train = Dataset("./TrainDataset_Aircheck.parquet").get_dataframe()

# **Custom Sampling for Training Data**
**Skip this step if you don't want to change the train set !**

In [ ]:
df_train = Dataset("./TrainDataset_Aircheck.parquet").get_dataframe()

# Define the ratio of negative samples (N times more than positives)
N = 2  # Adjust this value as needed
# Select all rows where DELLabel == 1 (positive samples)
positive_samples = df_train[df_train["DELLabel"] == 1]


# Select N times more rows where DELLabel == 0 (negative samples)
negative_samples = df_train[df_train["DELLabel"] == 0].sample(n=len(positive_samples) * N, random_state=42)

# Combine both subsets to create a balanced dataset with the desired ratio
df_balanced = pd.concat([positive_samples, negative_samples])

df_train=df_balanced

### **Development Dataset**

In [ ]:
DevData_path = './DevDataset_Aircheck.csv'

# Load the CSV file into a Pandas DataFrame
df_dev = Dataset(DevData_path).get_dataframe()
df_dev.head()

In [ ]:
print('Number of binders:',(df_dev['Label'] == 1).sum())
print('Number of non-binders:',(df_dev['Label'] == 0).sum())
df_dev.head(3)

### **Test Dataset**

In [ ]:

# Ensure that the file path is correct before running
TestData_path = './TestDataset_Aircheck.csv'

df_test = Dataset(TestData_path).get_dataframe()
print('Number of test compounds', len(df_test))
df_test.head(3)

In [ ]:
# Display first few rows of the test dataset
print('Number of test compounds', len(df_test))
df_test.head(3)

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_test.columns.tolist()
print(column_names_list)



---



# **🟣 3: Selecting Desired Columns (Finger prints) and ML labels**



In [ ]:

fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
selected_fps = 'ECFP4'  # Replace with desired fingerprints

# Create TrainData and TestData using the selected fingerprints

TrainData = ProcessData(df_train, selected_fps).get_data()
TestData = ProcessData(df_test, selected_fps).get_data()
DevData = ProcessData(df_dev, selected_fps).get_data()

# Creat TrainLabel from the 'DELLabel' column
TrainLabel = df_train['DELLabel']
DevLabel = df_dev['Label']





---



# **🟣 4: Define the ML Model**

**LightGBM**

Light Gradient Boosting Machine is a highly efficient, scalable machine learning framework for gradient boosting, designed for speed and performance. It builds decision trees by optimizing the model through iterative training, focusing on minimizing errors in predictions. Unlike traditional gradient boosting algorithms, LightGBM uses a histogram-based approach that speeds up the training process, especially for large datasets. It is known for handling categorical features directly, offering better accuracy with less computational cost. LightGBM is widely used for classification, regression, and ranking tasks, making it popular in data science competitions and real-world applications.

In [ ]:
from lightgbm import LGBMClassifier

# Initialize model with detailed hyperparameters using default values
model = LGBMClassifier(
    n_estimators=100,  # Number of boosting iterations (trees)
    n_jobs=1,  # Number of parallel jobs (1 for no parallelism)
    learning_rate=0.1,  # Learning rate
    max_depth=-1,  # No limit on maximum depth of trees
    min_samples_leaf=20,  # Minimum samples at leaf node
    min_samples_split=2,  # Minimum samples to split node
    lambda_l2=0.0,  # L2 regularization (no regularization)
    lambda_l1=0.0,  # L1 regularization (no regularization)
    num_leaves=31,  # Number of leaves in each tree
    max_bin=255,  # Maximum number of bins
    subsample=1.0,  # Subsample ratio for training data (use all data)
    colsample_bytree=1.0,  # Subsample ratio for features (use all features)
    use_best_model=True,  # Use the best model based on validation performance
    random_state=None,  # Random seed for reproducibility (None for random)
    boosting_type='gbdt',  # Boosting type (Gradient Boosting Decision Tree)
    early_stopping_rounds=None,  # No early stopping
    min_split_gain=0.0,  # Minimum loss reduction required to make a further partition
    ignore_column_check=False  # Do not handle missing values automatically
)

# Model is now initialized with default hyperparameters




---



# **🟣 5: Training the Model and Evaluating Performance on the Cross-Validation Set**

###**Cross Validation**

In [ ]:
import warnings
warnings.simplefilter("ignore", UserWarning)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)

# Function to train the model and compute all classification metrics
def train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test):
    """Train a LightGBM model and compute accuracy, precision, recall, F1, AUC, MCC, and Kappa."""
    model = LGBMClassifier(random_state=42)
    model.fit(CrossVal_data_train, CrossVal_label_train)

    y_pred = model.predict(CrossVal_data_test)
    y_scores = model.predict_proba(CrossVal_data_test)[:, 1]  # Probability for positive class

    metrics = {
        "Accuracy": accuracy_score(CrossVal_label_test, y_pred),
        "Precision": precision_score(CrossVal_label_test, y_pred, zero_division=0),
        "Recall": recall_score(CrossVal_label_test, y_pred),
        "F1-Score": f1_score(CrossVal_label_test, y_pred),
        "AUC-ROC": roc_auc_score(CrossVal_label_test, y_scores) if len(set(CrossVal_label_test)) > 1 else None,
        "MCC": matthews_corrcoef(CrossVal_label_test, y_pred),
        "Cohen's Kappa": cohen_kappa_score(CrossVal_label_test, y_pred),
    }

    return model, metrics

# Cross-validation
Nfold = 2
TrainData_np = np.array(TrainData)  # Ensure NumPy array
TrainLabel_np = np.array(TrainLabel)
skf = StratifiedKFold(n_splits=Nfold, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(TrainData_np, TrainLabel_np)):
    CrossVal_data_train, CrossVal_data_test = TrainData_np[train_idx], TrainData_np[test_idx]
    CrossVal_label_train, CrossVal_label_test = TrainLabel_np[train_idx], TrainLabel_np[test_idx]

    # Now using the renamed variables for both train and test data
    _, metrics = train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test)
    fold_metrics.append(metrics)

    # Print fold metrics
    print(f"Fold {fold_idx+1} Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print("-" * 40)

In [ ]:
# Compute average metrics across folds
avg_metrics = {metric: np.mean([fold[metric] for fold in fold_metrics]) for metric in fold_metrics[0]}

# Print average metrics
print("\nAverage Metrics across all folds:")
for metric, value in avg_metrics.items():
    print(f"{metric}: {value:.4f}")

---

### **Train Final Model**



In [ ]:

# Function to train the final model on the entire dataset
def train_final_model(X, y):
    final_model = LGBMClassifier(random_state=42)
    final_model.fit(X, y)
    return final_model

# Training the Final Model on the Full Dataset
final_model = train_final_model(TrainData, TrainLabel)

### **Model Evaluation on the Development Set**

In [ ]:
def evaluate_model(model, X_test):
    """Evaluate the model on the test set and return predictions and probabilities."""
    y_scores = model.predict_proba(X_test)[:, 1]  # Probability for positive class
    return np.round(y_scores, 3)

predictions = evaluate_model(final_model, DevData)
print("Predictions:\n", predictions[1:10], '\n')

Thr=0.5
df_predictions_dev = pd.DataFrame({
    'SMILES':df_dev['SMILES'],
    'PredictedScore': predictions,
    'PredictedLabel': (predictions > Thr).astype(int)  # Convert boolean to int (1 or 0)
})
df_predictions_dev.head(5)

**Threshold Optimization on the Development Set**

In [ ]:
Thr=0.6
df_predictions_dev = pd.DataFrame({
    'SMILES':df_dev['SMILES'],
    'PredictedScore': predictions,
    'PredictedLabel': (predictions > Thr).astype(int)  # Convert boolean to int (1 or 0)
})
df_predictions_dev.head(5)

# Sort by PredictedScore in descending order
df_sorted = df_predictions_dev.sort_values(by="PredictedScore", ascending=False)

# Define function to compute precision, hits, and TIR
def compute_metrics(df_sorted, df_dev, top_n):
    """Compute precision, number of hits, and true identification rate (TIR) for top N predictions."""
    top_n_df = df_sorted.head(top_n)
    merged_n = top_n_df.merge(df_dev, on="SMILES")

    # Precision: Correct predictions / total predictions
    precision = (merged_n["PredictedLabel"] == merged_n["Label"]).mean()

    # Number of Hits: Count of correctly identified positives (TPs)
    hits = ((merged_n["PredictedLabel"] == 1) & (merged_n["Label"] == 1)).sum()

    # True Identification Rate (TIR): Hits / Total Actual Positives in dev set
    total_actual_positives = (df_dev["Label"] == 1).sum()
    tir = hits / total_actual_positives if total_actual_positives > 0 else 0  # Avoid division by zero

    return precision, hits, tir

# Compute metrics at different thresholds
precision_100, hits_100, tir_100 = compute_metrics(df_sorted, df_dev, 100)
precision_50, hits_50, tir_50 = compute_metrics(df_sorted, df_dev, 50)
precision_20, hits_20, tir_20 = compute_metrics(df_sorted, df_dev, 20)

# Print results
print(f"Precision at Top 100: {precision_100:.4f}, Hits: {hits_100}, TIR: {tir_100:.4f}")
print(f"Precision at Top 50:  {precision_50:.4f}, Hits: {hits_50}, TIR: {tir_50:.4f}")
print(f"Precision at Top 20:  {precision_20:.4f}, Hits: {hits_20}, TIR: {tir_20:.4f}")

---

# 🚀**Virtual Screening**🧬


In [ ]:
display_google_drive_image('1OZDhx1wOLYu4Xy5CNZFsY7xntVe3mgmh')

# **🟣 6: Screen Test Compounds**
We can now screen new compounds to predict their activity. This involves passing SMILES strings through the trained model.

In [ ]:
def evaluate_model(model, X_test):
    """Evaluate the model on the test set and return predictions and probabilities."""
    y_scores = model.predict_proba(X_test)[:, 1]  # Probability for positive class
    return np.round(y_scores, 3)

predictions = evaluate_model(final_model, TestData)

# Create a DataFrame with SMILES and prediction scores
Thr=0.6
df_predictions_test = pd.DataFrame({
    'SMILES':df_test['SMILES'],
    'PredictedScore': predictions,
    'PredictedLabel': (predictions > Thr).astype(int)  # Convert boolean to int (1 or 0)
})

df_predictions_test.head(5)

**Selecting High-Scoring Predictions**

In [ ]:
Thr=0.5
# Sort the DataFrame by prediction score in descending order
prediction_df_sorted = df_predictions_test.sort_values(by='PredictedScore', ascending=False)

# Keep only those with score > Thr as Possible Nominees
nominees = prediction_df_sorted[prediction_df_sorted['PredictedScore'] > Thr]

# Get the number of nominees
num_nominees = nominees.shape[0]
print(f"\nNumber of Possible Nominees: {num_nominees}")

# Print the top 10 highest-ranked predictions
print("Top 10 Predictions:")
print(prediction_df_sorted.head(10))



---



# **🟣🟣🟣 How to submit results🟣🟣🟣**

In [ ]:
team_name = "Nabintest"
ResultSubmission.submit_result(team_name=team_name, df_predictions_test=df_predictions_test)

---

---



---



# **🟣 6: Using Ensemble of Models**

This method improves prediction reliability by training multiple models, each based on a different molecular fingerprint (e.g., ECFP4, FCFP6, MACCS). Instead of relying on a single model, we evaluate test data across all trained models, compute the mean prediction score, standard deviation, and confidence score. The final score, calculated as mean prediction minus standard deviation, helps select high-confidence nominees while reducing uncertainty. This approach ensures more robust and reliable predictions compared to using a single fingerprint-based model.

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

# List of fingerprint columns to train on
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
fingerprint_columns = ['ECFP4', 'MACCS', 'RDK', ]

# Dictionary to store trained models
trained_models = {}

# Train models for each fingerprint column
for fp in fingerprint_columns:
    print(f"Training model for {fp}...")
    TrainData = ProcessData(df_train, fp).get_data()
    TrainLabel = df_train['DELLabel']

    model = LGBMClassifier(random_state=42)
    model.fit(TrainData, TrainLabel)
    trained_models[fp] = model

# Function to evaluate models
def evaluate_model(model, X_test):
    return model.predict_proba(X_test)[:, 1]  # Probability for positive class

# Store predictions for each model
all_predictions = {}

for fp in fingerprint_columns:
    print(f"Evaluating model for {fp}...")
    TestData = ProcessData(df_test, fp).get_data()
    all_predictions[fp] = evaluate_model(trained_models[fp], TestData)

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(all_predictions)
predictions_df['Mean_Prediction'] = predictions_df.mean(axis=1)
predictions_df['Std_Dev'] = predictions_df.std(axis=1)
predictions_df['Confidence_Score'] = 1 - predictions_df['Std_Dev']  # Higher means more confident
predictions_df['Final_Score'] = predictions_df['Mean_Prediction'] - predictions_df['Std_Dev']

# Add SMILES column
predictions_df['SMILES'] = df_test['SMILES']

# Sort by Final Score
predictions_df_sorted = predictions_df.sort_values(by='Final_Score', ascending=False)

# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.5]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))



---



In [ ]:
# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.3]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))

# **🟣 7: Apply Medicinal Chemistry Filters**

This partapplies drug-likeness filters to the selected nominees based on three key rules: Lipinski, Ghose, and Veber. The molecules that pass all three filters are retained as final candidates.

- **Lipinski’s Rule of 5**: Evaluates drug-likeness based on molecular weight, lipophilicity (logP), hydrogen bond donors/acceptors, and rotatable bonds.
- **Ghose Filter**: Checks molecular weight, logP, number of atoms, and molar refractivity to ensure favorable pharmacokinetics.
- **Veber Rule**: Ensures compounds have limited rotatable bonds and acceptable topological polar surface area for good oral bioavailability.

**Note:** This is a simplified version of many available drug-like property filters!

In [ ]:
# Apply drug design filters to the selected nominees

drug_filter = SimplifiedDrugFilters()
filter_results = pd.DataFrame(drug_filter.filter(nominees["SMILES"].tolist()), index=nominees.index)

# Merge the filter results with nominees
nominees_filtered = pd.merge(nominees, filter_results, left_index=True, right_index=True)
print("Top 10 Predictions:")
print(nominees_filtered.head(10))

nominees_filtered = nominees_filtered[nominees_filtered["pass_all_filters"] == True]
num_nominees_filtered = nominees_filtered.shape[0]
print(f"\nNumber of Filtered Nominees: {num_nominees_filtered}")

---

# **🟣 8: Cluster and Select Nominated Compounds for Further Laboratory Test**
Finally, we use similarity-based clustering to identify diverse candidates among the top hits.

### 1. **Generate Molecular Fingerprints**:
   - Convert each molecule (represented by its SMILES string) into a **Morgan fingerprint** using RDKit's `AllChem.GetMorganFingerprintAsBitVect`. This produces a binary vector that encodes the molecular structure.

### 2. **Cluster Molecules Using LeaderPicker**:
   - Apply the **LeaderPicker** algorithm to the fingerprints to identify "leader" molecules. These leaders act as centroids for clusters of similar molecules. The `thresh` parameter determines the minimum similarity between a leader and other molecules for them to belong to the same cluster.

### 3. **Assign Molecules to Clusters**:
   - After identifying the leaders, calculate the **Tanimoto similarity** between each molecule's fingerprint and the fingerprints of the leader molecules. Each molecule is then assigned to the cluster whose leader it is most similar to. The function `assignPointsToClusters` groups molecules based on their similarity to these leaders.

### 4. **Select Representative Molecules and Sort**:
   - Within each cluster, select a subset of molecules (typically 1/20th of the cluster size). The selection is based on the similarity to previously selected molecules, ensuring diversity within the cluster. Finally, the results are sorted by **Prediction Score** and **Cluster ID** to highlight the most relevant molecules.


In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit import RDLogger
from rdkit import DataStructs
from rdkit.Chem import AllChem
from tqdm import tqdm

from rdkit import Chem
RDLogger.DisableLog('rdApp.*')


# Generate Morgan fingerprints using AllChem
fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), radius=3, nBits=2048) for smi in tqdm(nominees_filtered["SMILES"])]



In [ ]:
# Perform clustering using LeaderPicker
lp = rdSimDivPickers.LeaderPicker()
thresh = 0.65  # Minimum distance between cluster centroids
picks = lp.LazyBitVectorPick(fps, len(fps), thresh)  # Using the ExplicitBitVect fingerprints from AllChem
clusters = drug_filter.assignPointsToClusters(picks, fps)


# Assign cluster ids to the prediction_df based on the indices from the clusters
cluster_ids = np.zeros(len(nominees_filtered))  # Initialize cluster_ids for the entire prediction_df

# Make sure to correctly assign the cluster ids
for key, val in clusters.items():
    cluster_ids[val] = key  # Assign the cluster ID to the correct indices

# Add the cluster ids to the prediction_df
nominees_filtered['cluster_id'] = cluster_ids

# Sort the results by Prediction Score and Cluster ID
#nominees_filtered.sort_values(by=["Final_Score", "cluster_id"], ascending=[False, True], inplace=True)

num_clusters = len(set(nominees_filtered["cluster_id"]))
print(f"Number of clusters generated: {num_clusters}")

# Sort the dataframe by Final_Score in descending order
nominees_filtered.sort_values(by=["Final_Score"], ascending=False, inplace=True)

# Select one nominee per cluster: the one with the highest score
best_nominees = nominees_filtered.groupby("cluster_id").first().reset_index()

# Print the selected nominees (one per cluster)
print(best_nominees.head(10))



---

